# 5. Unified environmental focus dataset

This notebook combines the earlier build notebook and the `5a` analysis notebook into one consistent workflow. First it explains how the final CSV is built, then it analyzes the result so we can decide whether it is sensible for comfort prediction.


## What was done

The merge script combines seven CSV files into one row-level environmental dataset:

- `data/interim/keti_1min_resampled.csv`
- `data/raw/room_conditions.csv`
- `data/processed/1_room_measurements_with_comfort.csv`
- `data/raw/HomeCoach_5min_2023.csv`
- `data/raw/HomeCoach_5min_2024.csv`
- `data/raw/HomeCoach_5min_2025.csv`
- `data/raw/HomeCoach_5min_2026.csv`

The output is `data/processed/unified_environment_focus_dataset.csv`, plus `data/processed/unified_environment_focus_dataset_report.csv` for source-level quality checks.

The correct merge is a vertical union after standardizing column names. These files do not describe the same physical room at the same timestamp, so a timestamp join would create fake sensor rows.


## Why keep both `location_id` and `session_id`?

We use room, zone, sensor, student, or location columns to help create `location_id`, and then sessions are created inside that location. We keep both because they mean different things.

- `location_id`: stable grouping key, such as one room, zone, sensor, or student.
- `session_id`: a time-bounded block inside that grouping.
- One location can have many sessions.
- For ML, both are usually metadata at first. They help with grouping, leakage-safe train/test splits, quality checks, and deployment tracing.

The builder uses an existing session column if one exists. Otherwise it creates deterministic sessions from source, location, day changes, and long timestamp gaps.


In [ ]:
# Imports
from pathlib import Path
import sys

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ModuleNotFoundError:
    display = print

try:
    import matplotlib.pyplot as plt
    from matplotlib.patches import FancyArrowPatch, FancyBboxPatch
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "This notebook needs matplotlib. From the MAL folder, run: "
        "python -m pip install -r requirements.txt"
    ) from exc

plt.style.use("default")
pd.set_option("display.max_columns", 120)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")


In [ ]:
# Find the MAL folder whether this notebook is run from the repo root or from notebooks/data_related.
def find_mal_dir(start: Path) -> Path:
    for path in [start, *start.parents]:
        if path.name == "MAL" and (path / "data").exists():
            return path

        candidate = path / "MAL"
        if candidate.exists() and (candidate / "data").exists():
            return candidate

    raise RuntimeError("Could not find the MAL directory")


MAL_DIR = find_mal_dir(Path.cwd().resolve())
sys.path.insert(0, str(MAL_DIR))

DATA_PATH = MAL_DIR / "data" / "processed" / "unified_environment_focus_dataset.csv"
REPORT_PATH = MAL_DIR / "data" / "processed" / "unified_environment_focus_dataset_report.csv"

MAL_DIR, DATA_PATH.exists(), REPORT_PATH.exists()


## Input files and builder settings


In [ ]:
from scripts.build_unified_environment_dataset import (
    DEFAULT_OUTPUT_PATH,
    DEFAULT_REPORT_PATH,
    DEFAULT_SOURCES,
    OUTPUT_COLUMNS,
    build_unified_dataset,
)

source_files = pd.DataFrame(
    {
        "source": source.name,
        "path": source.path.relative_to(MAL_DIR),
        "exists": source.path.exists(),
    }
    for source in DEFAULT_SOURCES
)

display(source_files)
print("Output columns:")
print(OUTPUT_COLUMNS)


## Build or load the final CSV

Set `REBUILD_DATASET = True` if you want to rerun the combine step from scratch. By default the notebook loads the existing generated CSV and only builds it if it is missing.


In [ ]:
REBUILD_DATASET = False

if REBUILD_DATASET or not DATA_PATH.exists() or not REPORT_PATH.exists():
    final_df, report_df = build_unified_dataset()
else:
    final_df = pd.read_csv(DATA_PATH, parse_dates=["timestamp"], low_memory=False)
    report_df = pd.read_csv(REPORT_PATH, parse_dates=["first_timestamp", "last_timestamp"])

df = final_df
report = report_df

print(f"Dataset path: {DATA_PATH}")
print(f"Report path: {REPORT_PATH}")
print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns):,}")
print(f"Rows with focus_score: {df['focus_score'].notna().sum():,}")


## Column preview

Before deeper analysis, check the actual columns and a few rows. This is the quickest way to spot a wrong name, unexpected unit, or missing target.


In [ ]:
print("Columns:")
print(df.columns.tolist())

display(df.head(10))


In [ ]:
# Compact preview of the columns used for the first comfort-modeling experiments.
model_columns = ["humidity", "light", "temperature", "noise", "co2", "focus_score"]
sensor_columns = ["humidity", "light", "temperature", "noise", "co2"]

display(df[["timestamp", "source", "location_id", "session_id", *model_columns]].head(10))


## Final schema

The builder keeps the model-relevant sensor columns and adds traceability columns so rows can be grouped and audited later.


In [ ]:
schema_notes = pd.DataFrame([
    {"column": "timestamp", "meaning": "Measurement time, parsed and written consistently", "model_feature_now": False},
    {"column": "session_id", "meaning": "Time-bounded block inside a location", "model_feature_now": False},
    {"column": "location_id", "meaning": "Stable room, zone, student, device, or fallback grouping", "model_feature_now": False},
    {"column": "record_id", "meaning": "Stable row identifier with source prefix", "model_feature_now": False},
    {"column": "source", "meaning": "Dataset the row came from", "model_feature_now": False},
    {"column": "humidity", "meaning": "Relative humidity when available", "model_feature_now": True},
    {"column": "light", "meaning": "Light level when available", "model_feature_now": True},
    {"column": "temperature", "meaning": "Temperature when available", "model_feature_now": True},
    {"column": "noise", "meaning": "Noise level when available", "model_feature_now": True},
    {"column": "co2", "meaning": "CO2 level when available", "model_feature_now": True},
    {"column": "focus_score", "meaning": "Target label, mapped from comfort/rating columns", "model_feature_now": "target"},
])

display(schema_notes)


## ID relationship diagram

Raw location-like columns are normalized into `location_id`. Then `session_id` is created from existing session values when present, otherwise from location plus timestamp gaps.


In [ ]:
def draw_box(ax, xy, width, height, text, color):
    box = FancyBboxPatch(
        xy,
        width,
        height,
        boxstyle="round,pad=0.02,rounding_size=0.02",
        linewidth=1.2,
        edgecolor="#333333",
        facecolor=color,
    )
    ax.add_patch(box)
    ax.text(
        xy[0] + width / 2,
        xy[1] + height / 2,
        text,
        ha="center",
        va="center",
        fontsize=10,
        wrap=True,
    )


def draw_arrow(ax, start, end):
    arrow = FancyArrowPatch(
        start,
        end,
        arrowstyle="-|>",
        mutation_scale=18,
        linewidth=1.4,
        color="#333333",
    )
    ax.add_patch(arrow)


fig, ax = plt.subplots(figsize=(12, 4))
ax.set_xlim(0, 12)
ax.set_ylim(0, 4)
ax.axis("off")

draw_box(ax, (0.3, 1.25), 2.3, 1.4, "Raw identifiers\nroom / zone / sensor / student", "#edf2f7")
draw_box(ax, (3.2, 1.25), 2.1, 1.4, "location_id\nstable grouping key", "#e6fffa")
draw_box(ax, (6.0, 2.15), 2.3, 1.1, "timestamps\nand time gaps", "#fff5f5")
draw_box(ax, (6.0, 0.65), 2.3, 1.1, "existing session\nif source has one", "#fffaf0")
draw_box(ax, (9.0, 1.25), 2.4, 1.4, "session_id\ntime-bounded block", "#ebf8ff")

draw_arrow(ax, (2.6, 1.95), (3.2, 1.95))
draw_arrow(ax, (5.3, 1.95), (6.0, 2.55))
draw_arrow(ax, (5.3, 1.95), (6.0, 1.15))
draw_arrow(ax, (8.3, 2.55), (9.0, 1.95))
draw_arrow(ax, (8.3, 1.15), (9.0, 1.95))

ax.set_title("How identifiers are preserved and used", fontsize=14, pad=12)
plt.show()


# Analysis


## Source provenance and coverage


In [ ]:
coverage_columns = [
    "source",
    "raw_rows",
    "output_rows",
    "dropped_empty_rows",
    "labeled_rows",
    "unique_locations",
    "unique_sessions",
    "first_timestamp",
    "last_timestamp",
]

display(report[coverage_columns].sort_values("output_rows", ascending=False))


In [ ]:
source_summary = df.groupby("source").agg(
    rows=("record_id", "count"),
    labeled_rows=("focus_score", lambda values: int(values.notna().sum())),
    locations=("location_id", "nunique"),
    sessions=("session_id", "nunique"),
).sort_values("rows", ascending=False)

display(source_summary)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

source_summary["rows"].sort_values().plot(kind="barh", ax=axes[0], color="#3182ce")
axes[0].set_title("Rows by source")
axes[0].set_xlabel("Rows")
axes[0].set_ylabel("")

source_summary["labeled_rows"].sort_values().plot(kind="barh", ax=axes[1], color="#38a169")
axes[1].set_title("Rows with focus_score labels")
axes[1].set_xlabel("Labeled rows")
axes[1].set_ylabel("")

plt.tight_layout()
plt.show()


## Quick structural overview

This creates one small DataFrame per source, similar to the earlier mock-data notebooks.


In [ ]:
datasets = {
    f"Data {index}: {source}": source_df
    for index, (source, source_df) in enumerate(df.groupby("source", sort=True), start=1)
}

overview = pd.DataFrame({
    name: {
        "Rows": data_x.shape[0],
        "Columns": data_x.shape[1],
        "Missing values": int(data_x.isna().sum().sum()),
        "Locations": data_x["location_id"].nunique(),
        "Sessions": data_x["session_id"].nunique(),
        "Labeled rows": int(data_x["focus_score"].notna().sum()),
    }
    for name, data_x in datasets.items()
}).T

display(overview.sort_values("Rows", ascending=False))


## Simple range checks

These plain `describe()` checks help catch unit mismatches, broken imports, or values far outside the other sources.


In [ ]:
for name, data_x in datasets.items():
    print(f"{name} Ranges:")
    display(data_x[model_columns].describe().T)


In [ ]:
numeric_summary = []

for dataset_name, data_x in datasets.items():
    numeric_df = data_x.select_dtypes(include=np.number)

    for column in numeric_df.columns:
        numeric_summary.append({
            "Dataset": dataset_name,
            "Feature": column,
            "Min": numeric_df[column].min(),
            "Median": numeric_df[column].median(),
            "Mean": numeric_df[column].mean(),
            "Max": numeric_df[column].max(),
            "Std": numeric_df[column].std(),
            "Missing": int(numeric_df[column].isna().sum()),
        })

numeric_summary = pd.DataFrame(numeric_summary)

display(
    numeric_summary.sort_values(
        by=["Feature", "Dataset"]
    )
)


## Visual anomaly detection

The plots compare numeric feature distributions by source. Values are clipped to the 1st and 99th percentile per source before plotting so a few extreme readings do not hide the main shape.


In [ ]:
numeric_features = sorted(set(
    column
    for data_x in datasets.values()
    for column in data_x.select_dtypes(include=np.number).columns
))

for feature in numeric_features:
    fig, ax = plt.subplots(figsize=(10, 5))
    found = False

    for name, data_x in datasets.items():
        if feature not in data_x.columns:
            continue

        values = pd.to_numeric(data_x[feature], errors="coerce").dropna()
        if len(values) < 2:
            continue

        lower = values.quantile(0.01)
        upper = values.quantile(0.99)
        clipped = values.clip(lower, upper)
        ax.hist(clipped, bins=50, density=True, alpha=0.28, label=name)
        found = True

    if found:
        ax.set_title(f"Distribution comparison: {feature}")
        ax.set_xlabel(feature)
        ax.set_ylabel("Density")
        ax.legend()
        plt.tight_layout()
        plt.show()
    else:
        plt.close(fig)


## Missing value analysis

Missingness is expected because the public datasets do not all measure the same sensors. This is better than forcing fake values during the combine step.


In [ ]:
missing_percent = df[model_columns].isna().mean().mul(100).sort_values(ascending=False)
display(missing_percent.to_frame("missing_percent"))


In [ ]:
availability_by_source = df.groupby("source")[model_columns].apply(lambda part: part.notna().mean() * 100)

fig, ax = plt.subplots(figsize=(10, 5))
image = ax.imshow(availability_by_source.values, aspect="auto", cmap="YlGnBu", vmin=0, vmax=100)

ax.set_xticks(np.arange(len(model_columns)))
ax.set_xticklabels(model_columns, rotation=30, ha="right")
ax.set_yticks(np.arange(len(availability_by_source.index)))
ax.set_yticklabels(availability_by_source.index)
ax.set_title("Column availability by source (%)")

for row in range(availability_by_source.shape[0]):
    for col in range(availability_by_source.shape[1]):
        value = availability_by_source.iloc[row, col]
        ax.text(col, row, f"{value:.0f}", ha="center", va="center", fontsize=8)

fig.colorbar(image, ax=ax, label="Non-null percent")
plt.tight_layout()
plt.show()

display(availability_by_source.round(1))


## Sensor ranges

These checks show scale differences and possible outliers before preprocessing.


In [ ]:
range_summary = df.groupby("source")[sensor_columns].agg(["count", "min", "median", "mean", "max"])
display(range_summary)


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.ravel()

for ax, column in zip(axes, sensor_columns):
    values = df[column].dropna()
    if values.empty:
        ax.set_visible(False)
        continue

    lower = values.quantile(0.01)
    upper = values.quantile(0.99)
    values.clip(lower, upper).hist(bins=50, ax=ax, color="#4299e1")
    ax.set_title(f"{column} distribution\nclipped to 1st-99th percentile")
    ax.set_xlabel(column)
    ax.set_ylabel("Rows")

axes[-1].axis("off")
plt.tight_layout()
plt.show()


## Outliers (IQR Method)

This follows the same idea as `dataset_imputed_analysis.ipynb`: compute Q1, Q3, IQR, and count values outside `Q1 - 1.5 * IQR` and `Q3 + 1.5 * IQR`. These are candidates for review, not automatic deletion.


In [ ]:
numeric_df = df[model_columns].apply(pd.to_numeric, errors="coerce")

if numeric_df.empty:
    print("No numeric columns found for outlier analysis.")
else:
    outlier_stats = []
    for column in numeric_df.columns:
        series = numeric_df[column].dropna()
        if series.empty:
            outlier_stats.append({
                "column": column,
                "non_null_rows": 0,
                "lower_bound": np.nan,
                "upper_bound": np.nan,
                "outlier_count": 0,
                "outlier_pct": 0.0,
            })
            continue

        q1 = series.quantile(0.25)
        q3 = series.quantile(0.75)
        iqr = q3 - q1
        lower = q1 if iqr == 0 else q1 - 1.5 * iqr
        upper = q3 if iqr == 0 else q3 + 1.5 * iqr
        outliers = ((series < lower) | (series > upper)).sum()
        outlier_pct = (outliers / len(series)) * 100 if len(series) else 0

        outlier_stats.append({
            "column": column,
            "non_null_rows": len(series),
            "lower_bound": lower,
            "upper_bound": upper,
            "outlier_count": int(outliers),
            "outlier_pct": float(outlier_pct),
        })

    outlier_df = pd.DataFrame(outlier_stats).sort_values(by="outlier_pct", ascending=False)
    display(outlier_df)

    top_cols = outlier_df[outlier_df["outlier_count"] > 0].head(6)["column"].tolist()
    if not top_cols:
        print("No outliers detected by IQR rule.")
    else:
        plot_data = [numeric_df[column].dropna() for column in top_cols]
        fig, ax = plt.subplots(figsize=(10, 5))
        ax.boxplot(plot_data, vert=False, labels=top_cols, showfliers=True)
        ax.set_title("Outlier view by IQR candidate columns")
        ax.set_xlabel("Value")
        plt.tight_layout()
        plt.show()


In [ ]:
outlier_by_source = []

for source, source_df in df.groupby("source"):
    source_numeric = source_df[model_columns].apply(pd.to_numeric, errors="coerce")
    for column in source_numeric.columns:
        series = source_numeric[column].dropna()
        if series.empty:
            continue

        q1 = series.quantile(0.25)
        q3 = series.quantile(0.75)
        iqr = q3 - q1
        lower = q1 if iqr == 0 else q1 - 1.5 * iqr
        upper = q3 if iqr == 0 else q3 + 1.5 * iqr
        outliers = ((series < lower) | (series > upper)).sum()

        outlier_by_source.append({
            "source": source,
            "column": column,
            "non_null_rows": len(series),
            "outlier_count": int(outliers),
            "outlier_pct": (outliers / len(series)) * 100,
            "lower_bound": lower,
            "upper_bound": upper,
        })

outlier_by_source_df = pd.DataFrame(outlier_by_source)
display(outlier_by_source_df.sort_values("outlier_pct", ascending=False).head(20))


## Focus score labels

The current true supervised labels come from `1_room_measurements_with_comfort.csv`. Other sources are still useful for sensor distribution analysis and future imputation, but they are not labeled training examples yet.


In [ ]:
labeled_df = df[df["focus_score"].notna()].copy()
labeled_df["focus_score"] = labeled_df["focus_score"].astype(int)

print(f"Labeled rows: {len(labeled_df):,}")
display(labeled_df.groupby(["source", "focus_score"]).size().unstack(fill_value=0))
display(labeled_df.head(10))


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
labeled_df["focus_score"].value_counts().sort_index().plot(kind="bar", ax=ax, color="#38a169")
ax.set_title("Focus score distribution")
ax.set_xlabel("focus_score")
ax.set_ylabel("Rows")
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
plt.tight_layout()
plt.show()


## Correlation matrix for labeled rows

This is only a first diagnostic. The labeled dataset is small compared with the full sensor history, and correlations do not prove causation.


In [ ]:
corr_data = labeled_df[model_columns].dropna(axis=1, how="all")
corr = corr_data.corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(7, 6))
image = ax.imshow(corr.values, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(np.arange(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=45, ha="right")
ax.set_yticks(np.arange(len(corr.index)))
ax.set_yticklabels(corr.index)
ax.set_title("Correlation matrix on labeled rows")

for row in range(corr.shape[0]):
    for col in range(corr.shape[1]):
        value = corr.iloc[row, col]
        if pd.notna(value):
            ax.text(col, row, f"{value:.2f}", ha="center", va="center", fontsize=8)

fig.colorbar(image, ax=ax, label="Correlation")
plt.tight_layout()
plt.show()

display(corr)


### Interpreting the correlation values

Correlations like `humidity`/`temperature = 0.27` and `noise`/`co2 = 0.60` can be reasonable. The unified table is a vertical union of different rooms, years, sensors, and sampling rates, so global correlations include both physical relationships and source effects. A moderate `noise`/`co2` correlation is especially plausible because both can rise when occupancy increases. Use these numbers as diagnostics, not proof that one variable causes another.


In [ ]:
pairs_to_check = [("humidity", "temperature"), ("noise", "co2")]
pair_rows = []

for source, source_df in df.groupby("source"):
    for left, right in pairs_to_check:
        pair_data = source_df[[left, right]].dropna()
        pair_rows.append({
            "source": source,
            "pair": f"{left} / {right}",
            "rows_used": len(pair_data),
            "correlation": pair_data[left].corr(pair_data[right]) if len(pair_data) >= 2 else np.nan,
        })

pair_correlation_by_source = pd.DataFrame(pair_rows)
display(pair_correlation_by_source.sort_values(["pair", "source"]))


## Session and location analysis

This is where `location_id` and `session_id` both matter. If a location has many sessions, we can split by session or location to reduce leakage when evaluating models.


In [ ]:
session_summary = df.groupby("session_id").agg(
    source=("source", "first"),
    location_id=("location_id", "first"),
    rows=("record_id", "count"),
    first_timestamp=("timestamp", "min"),
    last_timestamp=("timestamp", "max"),
    labeled_rows=("focus_score", lambda values: int(values.notna().sum())),
)
session_summary["duration_minutes"] = (
    session_summary["last_timestamp"] - session_summary["first_timestamp"]
).dt.total_seconds() / 60

location_summary = df.groupby("location_id").agg(
    sources=("source", "nunique"),
    sessions=("session_id", "nunique"),
    rows=("record_id", "count"),
    first_timestamp=("timestamp", "min"),
    last_timestamp=("timestamp", "max"),
)

display(session_summary.sort_values("rows", ascending=False).head(10))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

session_summary["rows"].clip(upper=session_summary["rows"].quantile(0.99)).hist(
    bins=50,
    ax=axes[0],
    color="#805ad5",
)
axes[0].set_title("Rows per session\nclipped to 99th percentile")
axes[0].set_xlabel("Rows")
axes[0].set_ylabel("Sessions")

location_summary["sessions"].sort_values(ascending=False).head(15).sort_values().plot(
    kind="barh",
    ax=axes[1],
    color="#dd6b20",
)
axes[1].set_title("Top locations by number of sessions")
axes[1].set_xlabel("Sessions")
axes[1].set_ylabel("location_id")

plt.tight_layout()
plt.show()

display(location_summary.sort_values("sessions", ascending=False).head(10))


## Example time series

A quick look at one source/location helps confirm that timestamps and sessions are ordered correctly.


In [ ]:
example_source = "room_measurements_with_comfort" if "room_measurements_with_comfort" in set(df["source"]) else df["source"].iloc[0]
example_location = df.loc[df["source"].eq(example_source), "location_id"].mode().iloc[0]
example = df[df["source"].eq(example_source) & df["location_id"].eq(example_location)].sort_values("timestamp").head(300)

fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)

for ax, column, color in zip(axes, ["temperature", "co2", "focus_score"], ["#e53e3e", "#3182ce", "#38a169"]):
    if column in example and example[column].notna().any():
        ax.plot(example["timestamp"], example[column], marker="o", markersize=3, linewidth=1, color=color)
    ax.set_ylabel(column)
    ax.grid(True, alpha=0.25)

axes[0].set_title(f"Example time series: {example_source}, location_id={example_location}")
axes[-1].set_xlabel("timestamp")
plt.tight_layout()
plt.show()

display(example.head(10))


## ML conclusions

1. Use `focus_score.notna()` rows for the first supervised model.
2. Keep `session_id` and `location_id` out of the first sensor-only model features to avoid memorizing rooms or sessions.
3. Use `session_id` or `location_id` for train/test splitting so repeated measurements from the same room block do not leak across splits.
4. Handle missing `light` and `noise` in a preprocessing pipeline. Do not fill them during dataset combining.
5. Treat IQR outliers as candidates for review. Do not delete them blindly, because high CO2 or noise may be real and important for comfort prediction.
6. The unlabeled rows can still help with sensor range checks, drift monitoring, imputation experiments, and future pseudo-labeling.
